# Group 32 — verified multi-edit pilot
Rohith: setup, implementation, and verification. GPU verification is pending until the final export succeeds. Select a GPU runtime (prefer A100). Run cells in order. This notebook performs required verification after the plan’s November 21 cutoff; outputs record actual completion dates.

Stages: hardware → setup → Drive → 1 group → 5 groups → resume check → 20 groups in both orders → verify/export. No manual dataset upload is required.

In [ ]:
import sys, subprocess, shutil, json, os
from pathlib import Path
import torch
assert torch.cuda.is_available(), "Select a GPU runtime before proceeding."
print("Notebook Python:", sys.version.split()[0])
print("GPU:", torch.cuda.get_device_name(0))
print("GPU GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print("Free disk GiB:", round(shutil.disk_usage('/content').free / 2**30, 1))
assert shutil.disk_usage('/content').free > 20 * 2**30, "Need at least 20 GiB free."
def run(command, cwd=None):
    with subprocess.Popen([str(x) for x in command], cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
            if process.wait():
                raise RuntimeError("Command failed; inspect output above. Do not continue.")
        except KeyboardInterrupt:
            process.terminate()
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill(); process.wait()
            raise


## Install the pinned experiment environment
The notebook kernel may use Python 3.13; all model code runs in the separate Python 3.11 environment. The implementation checkout is pinned to a commit on first use and persisted in Drive below.

In [ ]:
REPO_URL = "https://github.com/Gogireddy007/Multifact-Rome.git"
BRANCH = "rohith/final-multi-edit"
ROOT = Path('/content/group32_final_multi')
if not ROOT.exists():
    run(['git', 'clone', '--branch', BRANCH, REPO_URL, ROOT])
assert not subprocess.check_output(['git','-C',str(ROOT),'status','--porcelain','--untracked-files=no'], text=True).strip(), "Checkout has local edits."
SOURCE_COMMIT = subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'], text=True).strip()
print("Experiment source:", SOURCE_COMMIT)
run([sys.executable, ROOT/'scripts/setup.py'], cwd=ROOT)
PYTHON = ROOT/'.venv/bin/python'
run([PYTHON, '-m', 'unittest', 'discover', '-s', 'scripts', '-p', 'test_*.py', '-v'], cwd=ROOT)


## Persistent output
Drive authorization may require your Google account confirmation. Keep the output name for resume; use a new name for intentional configuration/GPU changes. Stop if a previous run has a different commit; check out its recorded commit before resuming.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/Group32/ROME_final/pilot_both_v2')
OUT.mkdir(parents=True, exist_ok=True)
source_file = OUT/'source_commit.txt'
if source_file.exists():
    assert source_file.read_text().strip() == SOURCE_COMMIT, "Different source commit. Restore the recorded commit or use a new output folder."
else:
    source_file.write_text(SOURCE_COMMIT + '\n')
GROUPS = ROOT/'results/related_groups/pilot_groups.json'
def stage(n):
    run([PYTHON, ROOT/'scripts/run_multi_stage.py', '--groups', GROUPS,
         '--limit', n, '--order', 'both', '--output', OUT], cwd=ROOT)
    run([PYTHON, ROOT/'scripts/verify_multi_results.py', '--output', OUT, '--limit', n], cwd=ROOT)


## One-group smoke test: both orders

In [ ]:
stage(1)


## Five-group pilot: both orders

In [ ]:
stage(5)


## Resume check
This reruns verification only for the five completed groups. Result hashes must remain unchanged.

In [ ]:
import hashlib
before = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in OUT.glob('group_*.json')}
stage(5)
after = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in OUT.glob('group_*.json')}
assert before == after, "Resume changed completed results."
print("RESUME CHECK PASSED")


## Complete 20-group pilot
This resumes the first five groups and completes both orders for all 20. Progress appears at each edit/evaluation stage. Timings depend on the GPU; do not infer completion from elapsed time.

In [ ]:
stage(20)


## Verify and export
The ZIP is created only after re-verification. It contains raw prompt scores, independent controls, summary, CSV, logs, exact source, and environment metadata. The old score-only pilot remains separate.

In [ ]:
run([PYTHON, ROOT/'scripts/verify_multi_results.py', '--output', OUT, '--limit', 20], cwd=ROOT)
summary = json.loads((OUT/'summary_20.json').read_text())
assert summary['verified'] and summary['complete_all_groups'] and summary['num_group_orders'] == 40
assert len(list(OUT.glob('group_*.json'))) == 40
archive = shutil.make_archive('/content/Group32_Rohith_Final_Verified_Pilot', 'zip', OUT)
from google.colab import files
files.download(archive)
print("COMPLETE: 20 groups, 40 group/order results; verification passed.")
print("Saved:", archive)
